# Improved skill-catalogue forecasting

This notebook reads the training-only search and the frozen validation comparison. **Run All does not train, change the selection or evaluate the reserved test set.** Original results remain in `initial_v1` and notebook 09. The task still predicts the exact catalogue selection at +1, +2 and +5 seconds.

The frozen ensemble reaches **65.62%, 65.17% and 63.48% top-1 accuracy** at +1, +2 and +5 seconds, improving the original transformer by **6.65, 6.79 and 7.58 percentage points**. It uses 12 models and up to 60 seconds of history; the original used one model and five seconds. All ten pilots improve at every horizon; this does not imply improvement in every class or subset.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "bvr_behavior_prediction").is_dir())
sys.path.insert(0, str(ROOT))
BASE = ROOT / "artifacts/skill_forecasting"
DATA = BASE / "data_v1"
EXPERIMENT = BASE / "improvement_v3"
protocol = json.loads((EXPERIMENT / "protocol.json").read_text())
selection = json.loads((EXPERIMENT / "frozen_selection.json").read_text())
display(selection)

## Training-only selection

Each internal split assigns 147 training scenario groups to gradients and normalization, 21 to epoch selection/calibration, and 42 to candidate comparison. All ten pilot versions of a scenario stay together. The best candidates are checked on a second internal partition. The final configuration, epoch count, temperature and three seeds per component are frozen before the existing validation set is evaluated.

The original validation set was used in the initial experiment, so this remains development evidence. The reserved test is untouched.

In [ ]:
for directory in [BASE / "improvement_v2", EXPERIMENT]:
    print(directory.name)
    results = pd.read_csv(directory / "search_results.csv")
    display(results.sort_values(["fold", "mean_accuracy"], ascending=[True, False]).round(4))
    for fold in [0, 1]:
        split = json.loads((directory / f"inner_split_{fold}.json").read_text())
        print("Internal split", fold, pd.Series(split).value_counts().to_dict())

print("Fixed-mixture screen and two-partition confirmation")
display(pd.read_csv(EXPERIMENT / "ensemble_screen.csv").round(4))
display(pd.read_csv(EXPERIMENT / "ensemble_confirmation.csv").round(4))

## Matched validation comparison

All models are evaluated on the same forecast origins and horizon masks. Scores are window-weighted. Catalogue macro-F1 includes all 33 entries; supported macro-F1 includes labels with truth support. Individual seed results are shown to expose training variability; the ensemble was fixed as uniform probability averaging.

In [ ]:
comparison = pd.read_csv(EXPERIMENT / "comparison.csv")
columns = ["model", "horizon_s", "count", "top1_accuracy", "top3_accuracy",
           "macro_f1_catalogue", "macro_f1_supported", "nll"]
display(comparison[columns].round(4))
main = comparison[comparison.model.isin(["transformer", "gru", "training_frequency", "improved_ensemble"])]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
handles, labels = None, None
for ax, metric in zip(axes, ["top1_accuracy", "top3_accuracy", "macro_f1_catalogue"]):
    main.pivot(index="horizon_s", columns="model", values=metric).plot(kind="bar", ax=ax, rot=0)
    ax.set(title=metric.replace("_", " "), xlabel="Forecast horizon (s)", ylim=(0, 1))
    handles, labels = ax.get_legend_handles_labels()
    ax.get_legend().remove()
fig.legend(handles, labels, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.0))
plt.tight_layout(rect=(0, 0, 1, 0.88))
plt.show()

## Uncertainty from scenario groups

The paired bootstrap resamples complete scenario groups, preserving the dependence between all ten pilots and overlapping windows. These are descriptive development intervals, not final-test guarantees or a correction for all model selection.

In [ ]:
intervals = pd.read_csv(EXPERIMENT / "paired_scenario_bootstrap.csv")
display(intervals.assign(**{c: intervals[c] * 100 for c in ["accuracy_gain", "lower_95", "upper_95"]}))
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(intervals.horizon_s, intervals.accuracy_gain * 100,
            yerr=np.array([intervals.accuracy_gain - intervals.lower_95,
                           intervals.upper_95 - intervals.accuracy_gain]) * 100,
            fmt="o", capsize=5)
ax.axhline(0, color="grey", linewidth=1)
ax.set(xlabel="Forecast horizon (s)", ylabel="Top-1 accuracy gain (percentage points)",
       title="Improved ensemble versus original transformer\n95% scenario bootstrap intervals", xticks=[1, 2, 5])
plt.tight_layout()
plt.show()

## Per-pilot and changed-selection checks

Pilot identity is used for evaluation subsets only. Same/changed compares the future selection with the privileged true label at the origin, which is never an input. Endpoint equality does not rule out an intermediate switch and return.

In [ ]:
new_metrics = pd.read_csv(EXPERIMENT / "validation/ensemble/validation_metrics.csv")
old_metrics = pd.read_csv(BASE / "initial_v1/transformer/validation_metrics.csv")
metrics = pd.concat([old_metrics, new_metrics], ignore_index=True)
display(metrics.query("pilot_id != 'population' and cohort == 'all'").pivot(
    index=["pilot_id", "horizon_s"], columns="model", values="top1_accuracy").round(4))
display(metrics.query("pilot_id == 'population' and cohort in ['same', 'changed']")[
    ["model", "horizon_s", "cohort", "count", "top1_accuracy", "top3_accuracy", "macro_f1_catalogue"]].round(4))

## Class support and confusion matrices

Rare and absent labels remain in the catalogue. Row-normalized confusion matrices must be read together with support counts. Several catalogue names can execute the same guidance, limiting what can be inferred from physical observations.

In [ ]:
scores = pd.read_csv(EXPERIMENT / "validation/ensemble/per_class_metrics.csv")
display(scores.query("pilot_id == 'population'").sort_values(
    ["horizon_s", "support"], ascending=[True, False]).round(4))
matrices = np.load(EXPERIMENT / "validation/ensemble/confusion_matrices.npz")
names = matrices["classes"]
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for ax, horizon in zip(axes, [1, 2, 5]):
    counts = matrices[f"population_{horizon}s"]
    total = counts.sum(1, keepdims=True)
    values = np.divide(counts, total, out=np.zeros_like(counts, dtype=float), where=total > 0)
    ax.imshow(values, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(len(names)), names, rotation=90, fontsize=6)
    ax.set_yticks(range(len(names)), names, fontsize=6)
    ax.set(title=f"Improved ensemble, +{horizon}s", xlabel="Predicted", ylabel="True")
plt.tight_layout()
plt.show()

## Observation-only ensemble inference

Supply chronological live observations, excluding the reset frame. Short histories mask unavailable older tokens. Full 20-second and 60-second models need 206 and 606 raw frames respectively, including the acceleration lookback. No skill records, pilot IDs or future observations are supplied to the predictor.

In [ ]:
import torch
from bvr_behavior_prediction.skill_forecasting.improvement import ImprovedPredictor

torch.set_num_threads(4)

paths = [EXPERIMENT / "refit" / f"{component['candidate']}_s{seed}" / "best.pt"
         for component in selection["components"] for seed in component["seeds"]]
predictor = ImprovedPredictor(paths)
samples = pd.read_parquet(DATA / "samples.parquet")
origin = samples.loc[samples.split.eq("validation") & samples.step.ge(650)].iloc[0]
manifest = json.loads((DATA / "data_manifest.json").read_text())
observations = pd.read_parquet(
    Path(manifest["corpus"]) / "observations" / f"{origin.pilot_id}.parquet",
    filters=[("episode_id", "==", origin.episode_id), ("step", ">", 0), ("step", "<=", int(origin.step))])
raw = observations.sort_values("step")[[f"obs_{i:02}" for i in range(51)]].to_numpy()
forecast = predictor.predict(raw)
display(pd.DataFrame(forecast["top3_skills"], index=forecast["horizons_s"],
                     columns=["First", "Second", "Third"]).rename_axis("Horizon (s)"))
display(json.loads((EXPERIMENT / "verification.json").read_text()))
print("Report:", EXPERIMENT / "validation/report.html")

See [the experiment guide](../docs/skill_forecasting_improvements.md) for the complete protocol, commands, limitations and saved artifacts. These results concern new scenarios for the known pilot population; unseen-pilot performance remains untested.